# MLP с эмбеддингами: baseline

Третья модель к `catboost_baseline.ipynb` и `harmonic_baseline.ipynb`. Задача, метрика и
фолды те же, различается способ задать профиль:

| | как задаётся форма суток |
|---|---|
| CatBoost | ячейки `route x hour` нарезаются разбиениями |
| Гармоническая | сумма синусов периодов 24 и 168 часов |
| MLP | произведение обучаемых эмбеддингов route и часа недели |

Эмбеддинг часа недели учится совместно по всем маршрутам, поэтому профиль выходит гладким,
а редкие ячейки (ночь, праздники) не остаются с горсткой наблюдений. Для ансамбля важно
именно это - ошибки ложатся в других местах.

Код модели в `mlp.py`, лосс - L1 на сырых посадках, то есть ровно числитель WAPE.

## 1. Данные

In [1]:
import json
from datetime import date, datetime
from pathlib import Path

import numpy as np
import polars as pl
import plotly.express as px
import torch

from data_preparation import load_calendar, load_labels, enrich_features, build_submit
from metrics import wape, wape_score, wape_report
from backtest import make_folds, cv_score, fit_calibration, apply_calibration
from mlp import MLPSpec, fit_mlp, predict_mlp, save_mlp, resolve_device

In [2]:
ROUTES: tuple[int, ...] = (1, 5, 7, 11, 12, 17, 25, 26, 28, 50)

calendar = load_calendar("input/calendar/2025.xml")

train_df = enrich_features(
    load_labels("dataset/labels/labels_day_train.csv", date(2025, 1, 1), date(2025, 8, 31), routes=ROUTES),
    calendar,
)
test_df = enrich_features(
    load_labels("dataset/labels/labels_day_test.csv", date(2025, 9, 1), date(2025, 10, 31), routes=ROUTES),
    calendar,
)
submit_df = enrich_features(build_submit(routes=ROUTES), calendar)
all_df = pl.concat([train_df, test_df]).sort(["date", "hour", "route"])

for name, df, days in (("train", train_df, 243), ("test", test_df, 61), ("submit", submit_df, 61)):
    expected = days * 24 * len(ROUTES)
    assert df.height == expected, f"{name}: ожидалось {expected} строк, получено {df.height}"
    print(f"{name:>6}: {df.height:>6} строк, {df['date'].min()} - {df['date'].max()}")

dead_keys = (
    all_df.group_by(["route", "hour"]).agg(pl.col("target").sum().alias("total"))
    .filter(pl.col("total") == 0).select(["route", "hour"])
)
print(f"\nмёртвых пар route x hour: {dead_keys.height} (маршрут 5 целиком плюс глухие ночные часы)")
print(f"устройство: {resolve_device()}")

 train:  58320 строк, 2025-01-01 - 2025-08-31
  test:  14640 строк, 2025-09-01 - 2025-10-31
submit:  14640 строк, 2025-11-01 - 2025-12-31

мёртвых пар route x hour: 26 (маршрут 5 целиком плюс глухие ночные часы)
устройство: mps


## 2. Опорная модель

Конфигурация по умолчанию из `MLPSpec`. Обучение янв-авг, честная оценка на сен-окт.
Одна сеть учится 2-6 секунд, поэтому подбор здесь дешевле, чем у бустинга.

In [3]:
spec = MLPSpec(epochs=60)
print(spec)

state = fit_mlp(train_df, spec)
n_params = sum(p.numel() for p in state["models"][0].parameters())
print(f"обучено сетей: {len(state['models'])}, параметров в каждой: {n_params:,}")
print(f"маршрутов в модели: {len(state['route_to_idx'])} (маршрут 5 пропущен - нулевая история)")

pred_test = predict_mlp(state, test_df)
y_test = test_df["target"].to_numpy()

print(f"\nсен-окт: WAPE = {wape(y_test, pred_test):.4f}   WAPE-score = {wape_score(y_test, pred_test):.4f}")
display(wape_report(y_test, pred_test, test_df["route"], "route"))
display(wape_report(y_test, pred_test, test_df["hour"], "hour"))

MLPSpec(d_route=8, d_how=16, d_daytype=4, hidden=256, dropout=0.05, lr=0.003, weight_decay=0.0001, epochs=60, batch_size=2048, n_seeds=1, daylight=True, device='auto')
обучено сетей: 1, параметров в каждой: 43,733
маршрутов в модели: 9 (маршрут 5 пропущен - нулевая история)

сен-окт: WAPE = 0.1102   WAPE-score = 0.8898


route,y_sum,p_sum,err_sum,wape_local,wape_contrib,volume_share,bias
i32,f64,f64,f64,f64,f64,f64,f64
50,1.15746e6,1.3462e6,275013.295628,0.237601,0.021562,0.090749,0.163078
7,1.350573e6,1.4402e6,211316.961371,0.156465,0.016568,0.10589,0.066396
17,3.063025e6,3.0079e6,194655.752987,0.06355,0.015262,0.240153,-0.017987
11,1.907061e6,1.9083e6,175159.626681,0.091848,0.013733,0.149521,0.00067
12,1.983604e6,2.0219e6,170375.4033,0.085892,0.013358,0.155522,0.019293
26,1.089964e6,1.0205e6,109474.810065,0.100439,0.008583,0.085457,-0.06372
1,1.136093e6,1.0948e6,99534.395435,0.087611,0.007804,0.089074,-0.036377
25,464537.0,397669.568863,85902.543703,0.184921,0.006735,0.036421,-0.143944
28,602164.0,595023.561458,83823.609487,0.139204,0.006572,0.047212,-0.011858


hour,y_sum,p_sum,err_sum,wape_local,wape_contrib,volume_share,bias
i8,f64,f64,f64,f64,f64,f64,f64
8,1.145598e6,1.0924e6,112150.663361,0.097897,0.008793,0.089819,-0.046446
18,1.067295e6,1.0847e6,106730.803558,0.100001,0.008368,0.08368,0.016282
17,1.050353e6,1.0518e6,102499.936249,0.097586,0.008036,0.082352,0.00138
14,780547.0,777499.395142,100069.891846,0.128205,0.007846,0.061198,-0.003904
16,906936.0,911264.949799,98265.562042,0.108349,0.007704,0.071107,0.004773
…,…,…,…,…,…,…,…
0,25507.0,24403.797306,7414.68161,0.290692,0.000581,0.002,-0.043251
1,2465.0,1348.290483,2034.762403,0.825461,0.00016,0.000193,-0.453026
4,2304.0,1595.332713,1964.41229,0.85261,0.000154,0.000181,-0.307581


In [4]:
check = (
    test_df
    .with_columns(pl.Series("прогноз", pred_test))
    .filter((pl.col("route") == 17) & (pl.col("date") <= date(2025, 9, 14)))
    .with_columns(
        (pl.col("date_str") + pl.lit(" ") + pl.col("hour").cast(pl.String).str.zfill(2) + pl.lit(":00"))
        .str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M").alias("datetime")
    )
    .select(["datetime", "target", "прогноз"]).rename({"target": "факт"})
    .unpivot(index="datetime", variable_name="серия", value_name="посадки")
)
fig = px.line(check, x="datetime", y="посадки", color="серия",
              title="Маршрут 17, первые две недели сентября: факт против MLP")
fig.show()

## 3. Бэктест и вопрос калибровки

Фолды общие для всех трёх ноутбуков, из `backtest.py`. Калибровка внутри фолда считается
честно: коэффициенты снимаются на периоде, который предшествует валидации, по модели,
обученной ещё раньше. Это ровно та схема, что уходит в сабмит.

Проверяем оба варианта и выбираем по фолдам, а не по тому, насколько коэффициенты
отличаются от единицы - последнее считается на тех же данных и всегда льстит калибровке.

In [5]:
folds = make_folds(all_df)
for f in folds:
    print(f.describe())

fit_fn = lambda trn, cut: fit_mlp(trn, spec, cut)

mean_off, per_fold_off, _ = cv_score(fit_fn, predict_mlp, folds, calibrate=False)
mean_on, per_fold_on, details = cv_score(fit_fn, predict_mlp, folds, calibrate=True)

print(f"\nбез калибровки: CV WAPE = {mean_off:.4f}  {[round(s, 4) for s in per_fold_off]}")
print(f"с калибровкой:  CV WAPE = {mean_on:.4f}  {[round(s, 4) for s in per_fold_on]}")
for d in details:
    ks = d["calibration"]
    print(f"   фолд {d['cut']}: k от {min(ks.values()):.3f} до {max(ks.values()):.3f}")

USE_CALIBRATION = mean_on < mean_off
print(f"\nкалибровка {'включена' if USE_CALIBRATION else 'выключена'} - решение по фолдам")

край трейна 2025-04-30: train  28800 строк, valid 14640 (2025-05-01 - 2025-06-30), калибровка по 14640 строкам (2025-03-01 - 2025-04-30)
край трейна 2025-06-30: train  43440 строк, valid 14880 (2025-07-01 - 2025-08-31), калибровка по 14880 строкам (2025-04-30 - 2025-06-30)
край трейна 2025-08-31: train  58320 строк, valid 14640 (2025-09-01 - 2025-10-31), калибровка по 14640 строкам (2025-07-02 - 2025-08-31)

без калибровки: CV WAPE = 0.1753  [0.1619, 0.2538, 0.1101]
с калибровкой:  CV WAPE = 0.1760  [0.1822, 0.1765, 0.1694]
   фолд 2025-04-30: k от 0.911 до 1.145
   фолд 2025-06-30: k от 0.850 до 1.000
   фолд 2025-08-31: k от 0.850 до 1.000

калибровка выключена - решение по фолдам


## 4. Подбор

Перебираются поля `MLPSpec`. Во время подбора `n_seeds=1`, финальная модель усредняет
несколько сидов - разброс между ними заметный, а стоит это секунды.

In [6]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TRIALS = 25


def suggest(trial) -> MLPSpec:
    return MLPSpec(
        d_route=trial.suggest_categorical("d_route", [4, 8, 16]),
        d_how=trial.suggest_categorical("d_how", [8, 16, 32]),
        hidden=trial.suggest_categorical("hidden", [128, 256, 512]),
        dropout=trial.suggest_float("dropout", 0.0, 0.2),
        lr=trial.suggest_float("lr", 5e-4, 1e-2, log=True),
        weight_decay=trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        epochs=trial.suggest_int("epochs", 30, 150, step=10),
        batch_size=trial.suggest_categorical("batch_size", [1024, 2048, 4096]),
        daylight=trial.suggest_categorical("daylight", [True, False]),
        cyclic=trial.suggest_categorical("cyclic", [True, False]),
        cyclic_year=trial.suggest_categorical("cyclic_year", [True, False]),
        n_seeds=1,
    )


def objective(trial) -> float:
    trial_spec = suggest(trial)
    mean_wape, per_fold, _ = cv_score(
        lambda trn, cut: fit_mlp(trn, trial_spec, cut), predict_mlp, folds, calibrate=USE_CALIBRATION,
    )
    for i, s in enumerate(per_fold):
        trial.set_user_attr(f"fold{i}_wape", s)
    return mean_wape


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=10),
    study_name="mlp_wape",
)
study.enqueue_trial({
    "d_route": 8, "d_how": 16, "hidden": 256, "dropout": 0.05, "lr": 3e-3,
    "weight_decay": 1e-4, "epochs": 60, "batch_size": 2048, "daylight": True, "cyclic": False, "cyclic_year": False,
})
study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, show_progress_bar=True)

best = study.best_trial
best_spec = suggest(optuna.trial.FixedTrial(best.params))
print(f"CV WAPE = {best.value:.4f}  ->  WAPE-score = {max(0.0, 1 - best.value):.4f}")
print("по фолдам:", {k: round(v, 4) for k, v in best.user_attrs.items()})
print(best_spec)

display(study.trials_dataframe(attrs=("number", "value", "user_attrs", "params")).sort_values("value").head(10))

  0%|          | 0/25 [00:00<?, ?it/s]

CV WAPE = 0.1715  ->  WAPE-score = 0.8285
по фолдам: {'fold0_wape': 0.133, 'fold1_wape': 0.2648, 'fold2_wape': 0.1169}
MLPSpec(d_route=16, d_how=16, d_daytype=4, hidden=256, dropout=0.03731401177720717, lr=0.007247964111811321, weight_decay=0.0001436709513866422, epochs=130, batch_size=1024, n_seeds=1, daylight=False, device='auto')


,number,value,user_attrs_fold0_wape,user_attrs_fold1_wape,user_attrs_fold2_wape,params_batch_size,params_d_how,params_d_route,params_daylight,params_dropout,params_epochs,params_hidden,params_lr,params_weight_decay
7,7,0.171549,0.132979,0.264785,0.116883,1024,16,16,False,0.037314,130,256,0.007248,0.000144
24,24,0.171613,0.132766,0.265277,0.116796,1024,16,16,False,0.016371,140,128,0.004901,0.000371
23,23,0.171625,0.132956,0.264901,0.117019,1024,16,16,False,0.015505,140,128,0.007732,0.000050
22,22,0.171758,0.132864,0.265500,0.116909,1024,16,16,False,0.018037,120,128,0.007527,0.000039
21,21,0.171795,0.132463,0.265767,0.117154,1024,16,16,False,0.031721,120,128,0.007447,0.000037
17,17,0.171801,0.132857,0.265443,0.117102,1024,16,16,False,0.040359,90,128,0.006875,0.000006
15,15,0.171969,0.132843,0.265948,0.117115,1024,16,16,False,0.034987,120,128,0.007402,0.000019
18,18,0.172034,0.134631,0.264706,0.116764,1024,32,16,False,0.000083,90,128,0.004168,0.000005
10,10,0.172281,0.133116,0.265748,0.117977,1024,16,4,False,0.186458,150,256,0.009316,0.004272
13,13,0.172480,0.136303,0.264261,0.116876,1024,16,4,False,0.002008,130,256,0.005771,0.000024


## 5. Финальная модель

Обучение на всей истории янв-окт, усреднение по пяти сидам.

In [7]:
from dataclasses import replace

final_spec = replace(best_spec, n_seeds=5)
final_state = fit_mlp(all_df, final_spec)
print(f"финальная модель: {all_df.height} строк, {len(final_state['models'])} сидов, spec {final_spec}")

calib: dict[int, float] = {}
if USE_CALIBRATION:
    last = folds[-1]
    inner = fit_mlp(last.calib_train, final_spec, last.calib_cut)
    calib = fit_calibration(
        last.calib_valid["target"], predict_mlp(inner, last.calib_valid), last.calib_valid["route"],
    )
    print(f"калибровка: { {r: round(k, 3) for r, k in calib.items()} }")

ART = Path("artifacts/mlp")
ART.mkdir(parents=True, exist_ok=True)
save_mlp(final_state, ART / "model.pt")
with open(ART / "model_meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "model": "MLP with route and hour-of-week embeddings",
        "spec": final_spec.as_dict(),
        "routes": list(ROUTES),
        "train_period": ["2025-01-01", "2025-10-31"],
        "cv_wape": best.value,
        "cv_per_fold": dict(best.user_attrs),
        "use_calibration": bool(USE_CALIBRATION),
        "route_calibration": calib,
        "dead_route_hour_pairs": dead_keys.to_pandas().to_dict(orient="records"),
    }, f, indent=2, ensure_ascii=False)
print(f"сохранено в {ART}")

финальная модель: 72960 строк, 5 сидов, spec MLPSpec(d_route=16, d_how=16, d_daytype=4, hidden=256, dropout=0.03731401177720717, lr=0.007247964111811321, weight_decay=0.0001436709513866422, epochs=130, batch_size=1024, n_seeds=5, daylight=False, device='auto')
сохранено в artifacts/mlp


## 6. Submission

In [8]:
SUBMIT_DIR = Path("artifacts/submissions")
SUBMIT_DIR.mkdir(parents=True, exist_ok=True)
SUBMIT_PATH = SUBMIT_DIR / f"submission_mlp_{datetime.now().strftime('%m-%d-%Y_%H:%M:%S')}.csv"

submit_sorted = submit_df.sort(["date", "hour", "route"])
pred = predict_mlp(final_state, submit_sorted)
if USE_CALIBRATION:
    pred = apply_calibration(pred, submit_sorted["route"], calib)

dead = set(map(tuple, dead_keys.to_pandas().to_numpy()))
alive = np.array([(r, h) not in dead for r, h in zip(submit_sorted["route"].to_numpy(), submit_sorted["hour"].to_numpy())])
pred = np.where(alive, pred, 0.0)
print(f"занулено строк: {(~alive).sum()} из {len(alive)}")

submission = (
    pl.DataFrame({
        "route": submit_sorted["route"], "date": submit_sorted["date_str"],
        "hour": submit_sorted["hour"], "prediction": pred,
    })
    .with_columns(
        pl.col("route").cast(pl.Int32), pl.col("hour").cast(pl.Int32),
        pl.col("prediction").round(0).cast(pl.Int64),
    )
    .sort(["route", "date", "hour"])
)

expected_height = len(ROUTES) * 61 * 24
assert submission.height == expected_height, f"ожидалось {expected_height} строк, получено {submission.height}"
assert set(submission["route"].unique().to_list()) == set(ROUTES), "не все маршруты в сетке"
assert submission["date"].n_unique() == 61, "не все даты ноя-дек"
assert submission.select(pl.all().is_null().any()).row(0) == (False,) * 4, "есть пропуски"
assert submission["prediction"].min() >= 0, "есть отрицательные прогнозы"
assert submission.select(pl.struct(["route", "date", "hour"]).n_unique()).item() == submission.height, "дубли ключей"

submission.write_csv(SUBMIT_PATH, separator=";")
print(f"записано {submission.height} строк в {SUBMIT_PATH}")

display(all_df.group_by(pl.col("date").dt.month().alias("month")).agg(pl.col("target").sum().alias("факт")).sort("month").tail(3))
display(
    submission.with_columns(pl.col("date").str.to_date().dt.month().alias("month"))
    .group_by("month").agg(pl.col("prediction").sum().alias("прогноз")).sort("month")
)
print(f"итоговая сумма прогноза: {submission['prediction'].sum():,}")

занулено строк: 1586 из 14640
записано 14640 строк в artifacts/submissions/submission_mlp_09-25-2026_21:28:29.csv


month,факт
i8,i32
8,5309977
9,6147694
10,6606787


month,прогноз
i8,i64
11,5862514
12,6302473


итоговая сумма прогноза: 12,164,987


## 7. Прогнозы для ансамбля

Веса смеси подбираются по этим файлам - на фолдах, а не на цели.

In [9]:
# PRED_DIR = Path("artifacts/preds")
# PRED_DIR.mkdir(parents=True, exist_ok=True)
# MODEL_TAG = "mlp"

# for i, fold in enumerate(folds):
#     fitted = fit_mlp(fold.train, final_spec, fold.cut)
#     (
#         fold.valid.select(["date", "hour", "route", "target"])
#         .with_columns(pl.Series("pred", predict_mlp(fitted, fold.valid)))
#         .write_parquet(PRED_DIR / f"{MODEL_TAG}_fold{i}.parquet")
#     )

# submission.with_columns(pl.col("date").str.to_date()).rename({"prediction": "pred"}).write_parquet(
#     PRED_DIR / f"{MODEL_TAG}_submit.parquet"
# )

# for f in sorted(PRED_DIR.glob("*.parquet")):
#     print(f"  {f.name}  {f.stat().st_size / 1024:.0f} КБ")